In [3]:
pip install findspark
import findspark

In [4]:
findspark.find()

'C:\\spark-3.5.5-bin-hadoop3'

In [5]:
findspark.init()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lag, avg, stddev, count, monotonically_increasing_id
from pyspark.sql.window import Window
import os

spark = SparkSession.builder.appName("FeatureEngineering").getOrCreate()

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
df = spark.read.parquet(os.path.join(base_dir,r"blob_storage_simulation\silver\cleaned_data.parquet"))

In [3]:
# Encode "Type" column: H=2, L=0, M=1
df = df.withColumn("type_encoded", 
                   when(col("Type") == "L", 0)
                   .when(col("Type") == "M", 1)
                   .otherwise(2))

In [4]:

# One-hot encode failure type (for demonstration)
for failure in ["No Failure", "Heat Dissipation Failure", "Power Failure", "Overstrain Failure", "Random Failures", "Tool Wear Failure"]:
    col_name = failure.lower().replace(" ", "_")
    df = df.withColumn(col_name, when(col("Failure Type") == failure, 1).otherwise(0))

In [5]:
# ---- Time-series Features ----
# Tool wear as a proxy for time (monotonically increasing index)
windowSpec = Window.orderBy("Tool wear [min]")

# Rolling mean features
for feature in ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]"]:
    df = df.withColumn(f"{feature}_rolling_mean_3", avg(col(feature)).over(windowSpec.rowsBetween(-2, 0)))
    df = df.withColumn(f"{feature}_rolling_std_3", stddev(col(feature)).over(windowSpec.rowsBetween(-2, 0)))
    df = df.withColumn(f"{feature}_lag_1", lag(col(feature), 1).over(windowSpec))


In [8]:
# Rate of change (delta) features
for feature in ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]"]:
    df = df.withColumn(f"{feature}_delta", col(feature) - lag(col(feature), 1).over(windowSpec))

In [ ]:
# Cumulative average of Tool wear
df = df.withColumn("cumulative_tool_wear_avg", 
                   avg("Tool wear [min]").over(Window.orderBy("Tool wear [min]").rowsBetween(Window.unboundedPreceding, 0)))

In [9]:
# Target lag (can help in certain models, e.g., failure prediction)
df = df.withColumn("target_lag_1", lag("Target", 1).over(windowSpec))

In [10]:
# Save features
df.write.mode("overwrite").parquet(os.path.join(base_dir,r"blob_storage_simulation\features\final_features.parquet"))

In [12]:
# (Optional) If you want to convert to Pandas for ML model input
features_pd = df.toPandas()
print(features_pd.head())

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1   79     L47258    L                298.8                    308.9   
2  163     L47342    L                298.3                    308.1   
3  251     L47430    L                298.0                    308.3   
4  333     M15192    M                297.6                    308.3   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Target Failure Type  \
0                    1551         42.8                0       0   No Failure   
1                    1398         51.5                0       0   No Failure   
2                    1586         35.5                0       0   No Failure   
3                    1662         32.7                0       0   No Failure   
4                    1538         40.2                0       0   No Failure   

   ...  Rotational speed [rpm]_rolling_std_3  Rotational speed [rpm]_lag_1  \
0  ...  

In [13]:
df.write.csv(os.path.join(base_dir,r"notebooks\sample_features"), header=True, mode="overwrite")


In [14]:
spark.stop()